# Hierarchical AR with stochastic emission

The teacher runs the usual AR process on a *hidden* sequence $h_t$; each $h_t$ then emits a block of `chunk_size` observed sub-tokens. With `stochastic_emission=True`, each hidden id has a fixed set of `chunk_size` observed items and positions are sampled with replacement from that set — so different occurrences of the same $h$ emit different observed chunks.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np, torch, matplotlib.pyplot as plt
from src.model import VectorARModel
from src.data import HierarchicalGaussianARClassification

torch.manual_seed(0)

DIM, CHUNK_DIM, CHUNK_SIZE, WINDOW = 10, 6, 3, 2
SPAN_LENGTHS, LENGTH = [2, 2], 6

teacher = VectorARModel.from_parameters(dim=DIM, window=WINDOW, span_lengths=SPAN_LENGTHS)
ds = HierarchicalGaussianARClassification(
    teacher=teacher, window=WINDOW, dim=DIM,
    chunk_dim=CHUNK_DIM, chunk_size=CHUNK_SIZE,
    number=1, length=LENGTH,
    prefix_length=sum(SPAN_LENGTHS),
    unroll_sequences=False, one_hot=True,
    softmax=True, temperature=0.1,
    stochastic_emission=True,
)

## One hidden trajectory, two stochastic emissions

Top row: the hidden sequence (token ids in red). Bottom two rows: two independent emissions of that *same* trajectory. Dashed lines mark chunk boundaries — within each chunk, the student sees a different realization each time.

In [ ]:
hidden = ds.data[0]
hidden_ids = hidden.argmax(dim=-1).tolist()
obs_a, obs_b = ds[0].numpy(), ds[0].numpy()

fig, axes = plt.subplots(3, 1, figsize=(10, 4.5), gridspec_kw={'height_ratios': [0.6, 2, 2]})

ax = axes[0]
ax.imshow(hidden.T.numpy(), aspect='auto', cmap='Greys', interpolation='nearest')
for t, h in enumerate(hidden_ids):
    ax.text(t, h, str(h), ha='center', va='center', color='red', fontsize=9)
ax.set_yticks([]); ax.set_xticks(range(len(hidden_ids)))
ax.set_title(f'hidden $h_t$  (dim={DIM})')

for ax, obs, label in zip(axes[1:], [obs_a, obs_b], ['emission A', 'emission B']):
    ax.imshow(obs.T, aspect='auto', cmap='Greys', interpolation='nearest')
    ax.set_title(label)
    for b in range(1, len(hidden_ids)):
        ax.axvline(b * CHUNK_SIZE - 0.5, color='tab:blue', lw=0.8, ls='--')
    ax.set_xticks([i * CHUNK_SIZE + (CHUNK_SIZE - 1) / 2 for i in range(len(hidden_ids))])
    ax.set_xticklabels([f'{hidden_ids[i]}' for i in range(len(hidden_ids))])

plt.tight_layout(); plt.show()
print('A != B:', not np.array_equal(obs_a, obs_b))

## Expected attention structure

The student must do two things:

1. **Copy within chunk** — pool the $k$ sub-token positions that share a hidden state to recover $h$.
2. **Run AR on recovered states** — attend, at the chunk level, to the previous $W$ chunks.

In a causal attention matrix these are two distinct shapes: block-diagonal on the main diagonal vs. thin stripes at chunk starts $\leq W$ steps back.

In [ ]:
T = (LENGTH + len(SPAN_LENGTHS)) * CHUNK_SIZE
causal = np.tril(np.ones((T, T)))

def norm_rows(m):
    m = m * causal
    return m / (m.sum(-1, keepdims=True) + 1e-9)

copying = np.zeros((T, T))
for t in range(T):
    cs = (t // CHUNK_SIZE) * CHUNK_SIZE
    copying[t, cs:min(cs + CHUNK_SIZE, t + 1)] = 1.0

ar = np.full((T, T), 1e-3)
for t in range(T):
    for lag in range(1, WINDOW + 1):
        s = (t // CHUNK_SIZE - lag) * CHUNK_SIZE
        if s >= 0: ar[t, s] = 1.0

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, m, title in zip(axes, [norm_rows(copying), norm_rows(ar)],
                        ['copying head (within-chunk)', f'AR head (chunk-level, W={WINDOW})']):
    ax.imshow(m, cmap='Blues', vmin=0, vmax=1, aspect='equal')
    ax.set_title(title); ax.set_xlabel('key'); ax.set_ylabel('query')
    for b in range(CHUNK_SIZE, T, CHUNK_SIZE):
        ax.axhline(b - 0.5, color='k', lw=0.3, alpha=0.4)
        ax.axvline(b - 0.5, color='k', lw=0.3, alpha=0.4)
plt.tight_layout(); plt.show()

## Incremental learning

Three metrics, expected to rise in this order:

1. **Intra-chunk attention mass** (new): fraction of a head's attention that lands inside the current chunk. Signals the copying head.
2. **Attention-span alignment in chunk coordinates**: existing `log_attention_alignment` but with stride `chunk_size`. Signals the AR head.
3. **Value-matrix alignment with teacher $A_i$**: existing `log_value_matrix_alignment`, unchanged.

Baseline run: `dataset=classification/hierarchical` (deterministic) — metric (1) is trivial; (2) and (3) rise roughly together. Treatment run: `dataset=classification/hierarchical_stochastic` — (1) ≺ (2) ≺ (3) should show as three separated phase transitions. That ordering **is** the empirical test of two-stage computation.

Metric (1) requires one new function in `src/visualizer.py`; (2) and (3) already exist.